In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Settings

In [ ]:
# Set display options for pandas
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [ ]:
DATASET_NAME_ORDER = [
    "BH_1",
    "DA",
    "alkox",
    "oer_plate_a",
    "p3ht",
    "photo_pce10",
    "photo_wf3",
    "suzuki_edbo",
    "suzuki",
]
MODEL_LABELS_ORDER = [
    "GPT-5 mini medium",
    "GPT-4.1 mini temp1.0",
    "GPT-4o mini temp1.0",
    "Claude Sonnet 4.5 temp1.0",
    "Claude Haiku 4.5 temp1.0",
    "Claude 3.5 Haiku temp1.0",
]

## Analysis

In [ ]:
direct_scores_df = pd.read_csv("./direct_results_final/scores.csv")
pairwise_scores_df = pd.read_csv("./pairwise_results_final/scores.csv")

In [ ]:
# Compare direct vs pairwise correlations per dataset/model
metric_order = [
    "pearson_corr",
    "spearman_corr",
    "kt_corr",
]

# Prepare subsets with consistent naming
direct_scores_df = direct_scores_df[
    ["dataset_name", "model_w_params", "pearson_corr", "spearman_corr", "kt_corr"]
].rename(
    columns={
        "pearson_corr": "pearson_corr_direct",
        "spearman_corr": "spearman_corr_direct",
        "kt_corr": "kt_corr_direct",
    }
)
pairwise_scores_df = pairwise_scores_df[
    [
        "dataset_name",
        "model_w_params",
        "pearson_corr_original",
        "spearman_corr_original",
        "kt_corr_original",
        "pearson_corr_extracted",
        "spearman_corr_extracted",
        "kt_corr_extracted",
    ]
].rename(
    columns={
        "pearson_corr_original": "pearson_corr_pairwise_all",
        "spearman_corr_original": "spearman_corr_pairwise_all",
        "kt_corr_original": "kt_corr_pairwise_all",
        "pearson_corr_extracted": "pearson_corr_pairwise_extracted",
        "spearman_corr_extracted": "spearman_corr_pairwise_extracted",
        "kt_corr_extracted": "kt_corr_pairwise_extracted",
    }
)

comparison_df = direct_scores_df.merge(
    pairwise_scores_df, on=["dataset_name", "model_w_params"], how="left"
)

comparison_df = comparison_df.assign(
    dataset_name=pd.Categorical(
        comparison_df["dataset_name"], categories=DATASET_NAME_ORDER, ordered=True
    ),
    model_w_params=pd.Categorical(
        comparison_df["model_w_params"], categories=MODEL_LABELS_ORDER, ordered=True
    ),
).sort_values(["dataset_name", "model_w_params"])

In [ ]:
def plot_corr_coef(datasets, metrics):
    metrics_labels = {
        "pearson_corr": "Pearson \ncorrelation coefficient",
        "spearman_corr": "Spearman's rank \ncorrelation coefficient",
        "kt_corr": "Kendall's rank \ncorrelation coefficient",
    }
    filename_labels = {
        "pearson_corr": "pearson",
        "spearman_corr": "spearman",
        "kt_corr": "kendall",
    }

    ncol = 3
    nrow = len(datasets) // ncol + (len(datasets) % ncol > 0)
    fig, axes = plt.subplots(
        nrow, ncol, figsize=(ncol * 4, nrow * 3), sharex=True, sharey=True
    )
    axes = axes.flatten()
    handles, labels = None, None
    for i, ds in enumerate(datasets):
        sub = comparison_df[comparison_df["dataset_name"] == ds].sort_values(
            "model_w_params"
        )
        x = np.arange(len(sub))
        width = 0.2
        ax = axes[i]
        ax.bar(
            x - width,
            sub[f"{metrics}_pairwise_all"],
            width,
            label="Preference learning (all instances)",
            color="#9467bd",
        )
        ax.bar(
            x,
            sub[f"{metrics}_pairwise_extracted"],
            width,
            label="Preference learning (sampled instances)",
            color="#1f77b4",
        )
        ax.bar(
            x + width,
            sub[f"{metrics}_direct"],
            width,
            label="Direct prediction",
            color="#ff7f0e",
        )

        ax.grid(axis="y", linestyle="--", alpha=0.3)
        ax.axhline(0, color="black", linewidth=0.5)
        ax.set_ylim(-0.5, 1.0)
        ax.tick_params(axis="y", labelsize=10)
        if i % ncol == 0:
            ax.set_ylabel(metrics_labels[metrics], fontsize=12)
        else:
            ax.set_ylabel("")
            ax.tick_params(axis="y", labelleft=False)
        ax.set_xticks(x)
        ax.set_xticklabels(sub["model_w_params"], rotation=60, ha="right", fontsize=10)
        ax.set_title(ds, fontsize=12, y=0.875)
        if i == 0:
            handles, labels = ax.get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.04),
        fontsize=10,
        ncol=3,
        frameon=True,
    )
    fig.tight_layout()
    plt.savefig(
        f"./images/preference_learning_vs_direct_prediction_{filename_labels[metrics]}.png",
        format="png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.savefig(
        f"./images/preference_learning_vs_direct_prediction_{filename_labels[metrics]}.pdf",
        format="pdf",
        bbox_inches="tight",
    )
    plt.savefig(
        f"./images/preference_learning_vs_direct_prediction_{filename_labels[metrics]}.svg",
        format="svg",
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
plot_corr_coef(datasets=DATASET_NAME_ORDER, metrics="pearson_corr")
plot_corr_coef(datasets=DATASET_NAME_ORDER, metrics="spearman_corr")
plot_corr_coef(datasets=DATASET_NAME_ORDER, metrics="kt_corr")

In [ ]:
def plot_specific_corr_coef(datasets, metrics):
    metrics_labels = {
        "pearson_corr": "Pearson \ncorrelation coefficient",
        "spearman_corr": "Spearman's rank \ncorrelation coefficient",
        "kt_corr": "Kendall's rank \ncorrelation coefficient",
    }
    filename_labels = {
        "pearson_corr": "pearson",
        "spearman_corr": "spearman",
        "kt_corr": "kendall",
    }
    fig, axes = plt.subplots(
        len(datasets), 1, figsize=(5, len(datasets) * 3), sharex=True
    )
    axes = axes.flatten()
    handles, labels = None, None
    for i, ds in enumerate(datasets):
        sub = comparison_df[comparison_df["dataset_name"] == ds].sort_values(
            "model_w_params"
        )
        x = np.arange(len(sub))
        width = 0.25
        ax = axes[i]
        ax.bar(
            x - width / 2,
            sub[f"{metrics}_pairwise_extracted"],
            width,
            label="Preference learning",
            color="#1f77b4",
        )
        ax.bar(
            x + width / 2,
            sub[f"{metrics}_direct"],
            width,
            label="Direct prediction",
            color="#ff7f0e",
        )
        ax.grid(axis="y", linestyle="--", alpha=0.3)
        ax.axhline(0, color="black", linewidth=0.5)
        ax.set_ylim(-0.5, 1.0)
        ax.tick_params(axis="y", labelsize=10)
        ax.set_ylabel(metrics_labels[metrics], fontsize=12)
        ax.set_xticks(x)
        ax.set_xticklabels(sub["model_w_params"], rotation=60, ha="right", fontsize=10)
        ax.set_title(ds, fontsize=12, y=0.85)
        if i == 0:
            handles, labels = ax.get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.602, 1.06),
        fontsize=10,
        ncol=2,
        frameon=True,
    )
    fig.tight_layout()
    plt.savefig(
        f"./images/preference_learning_vs_direct_prediction_{filename_labels[metrics]}_part.png",
        format="png",
        dpi=600,
        bbox_inches="tight",
    )
    plt.savefig(
        f"./images/preference_learning_vs_direct_prediction_{filename_labels[metrics]}_part.pdf",
        format="pdf",
        bbox_inches="tight",
    )
    plt.savefig(
        f"./images/preference_learning_vs_direct_prediction_{filename_labels[metrics]}_part.svg",
        format="svg",
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
plot_specific_corr_coef(datasets=["suzuki", "photo_wf3"], metrics="pearson_corr")
plot_specific_corr_coef(datasets=["suzuki", "photo_wf3"], metrics="spearman_corr")
plot_specific_corr_coef(datasets=["suzuki", "photo_wf3"], metrics="kt_corr")